# 05C COF 高通量筛选：从训练模型到候选材料

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/05C_high_throughput_screening.ipynb)

05B 解决的是“如何从真实 CIF 构建 ML 数据”。这一章回答下一步：**训练完模型以后，材料机器学习究竟用来做什么？**

我们借鉴公开仓库 `jsdvos/SupportingInformation_CO2captureHTS_2024` 的科研路线。该项目把流程组织为 benchmark → ideal screening → machine learning → mixture screening → analysis，并在机器学习阶段显式使用 `features.csv`、`results.csv`、train/test structure lists 和 SHAP 分析。

本章不复制其大规模数据和全部脚本，而是把科研工作流缩小成初学者可运行的版本。

## 1. 研究级 COF screening 的基本结构

```text
COF library
   ↓
structure / pore / chemistry descriptors
   ↓
expensive reference calculations on a subset
   ↓
features + target results
   ↓
train / validation / test
   ↓
ML surrogate
   ↓
predict a much larger candidate library
   ↓
rank candidates → interpret → return to CIF → validate
```

重点不是“Random Forest 比哪个模型更强”，而是 **用少量昂贵计算训练 surrogate，再筛选大量候选结构**。

## 2. 与公开 COF CO₂ screening 仓库对应

可借鉴的仓库：`jsdvos/SupportingInformation_CO2captureHTS_2024`。其中 `Step2_MachineLearning` 使用 `features.csv`、`results.csv`、`structs_train.txt`、`structs_test.txt`，并包含 feature reduction、模型训练和 SHAP 分析。

这给教程一个重要启示：**真实科研项目很少只有一个 notebook。数据生成、模型、筛选和分析应彼此分离，但通过结构 ID 保持可追溯。**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
rng = np.random.default_rng(42)


## 3. 用缩小版 screening table 理解机制

为了让这一章独立运行，我们构造一个**教学 candidate table**。这些 target 是教学公式生成的，不能用于科研结论；真实项目应替换为统一条件下的 GCMC/实验结果。

05B 与 05C 的区别：05B 是 **真实 CIF → descriptor table**；05C 是 **reference subset → surrogate model → large-library screening**。两章合起来才是一条完整路线。

In [ ]:
n = 500
candidates = pd.DataFrame({
    'COF_ID': [f'candidate_{i:04d}' for i in range(n)],
    'PLD_A': rng.uniform(3.0, 18.0, n),
    'LCD_A': rng.uniform(5.0, 35.0, n),
    'ASA_m2_g': rng.uniform(200, 4500, n),
    'void_fraction': rng.uniform(0.25, 0.90, n),
    'density_g_cm3': rng.uniform(0.25, 1.40, n),
    'N_fraction': rng.uniform(0.0, 0.20, n),
    'O_fraction': rng.uniform(0.0, 0.20, n),
})
noise = rng.normal(0, 0.35, n)
candidates['CO2_uptake_demo'] = (
    0.0010*candidates['ASA_m2_g'] + 2.2*candidates['N_fraction']
    + 1.2*candidates['O_fraction'] + 0.8*candidates['void_fraction']
    - 0.045*np.abs(candidates['PLD_A']-7.0) + noise
)
candidates.head()


## 4. 昂贵 reference calculation 只做一部分

假设 500 个候选 COF 中，只负担得起 120 个 reference calculations。机器学习的价值在于从这 120 个学习结构–性质关系，再预测其余材料。

In [ ]:
feature_cols = ['PLD_A','LCD_A','ASA_m2_g','void_fraction','density_g_cm3','N_fraction','O_fraction']
reference = candidates.sample(120, random_state=42).copy()
unseen = candidates.drop(reference.index).copy()
X_train, X_test, y_train, y_test = train_test_split(reference[feature_cols], reference['CO2_uptake_demo'], test_size=0.25, random_state=42)
model = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
test_pred = model.predict(X_test)
print('MAE =', mean_absolute_error(y_test, test_pred))
print('R2  =', r2_score(y_test, test_pred))


## 5. 用 surrogate 筛选未计算的 COF

只有在 test/validation 表现合理之后，才把模型应用到没有 reference target 的 candidate library。

In [ ]:
unseen['predicted_CO2_uptake_demo'] = model.predict(unseen[feature_cols])
top20 = unseen.sort_values('predicted_CO2_uptake_demo', ascending=False).head(20)
display(top20[['COF_ID','predicted_CO2_uptake_demo'] + feature_cols])


## 6. 排名不是终点：检查 applicability domain

预测最高的材料可能位于训练数据范围之外。下面只做最简单的 min–max 检查；科研中可以进一步使用距离、不确定度、ensemble disagreement 等方法。

In [ ]:
train_min = reference[feature_cols].min()
train_max = reference[feature_cols].max()
outside = ((top20[feature_cols] < train_min) | (top20[feature_cols] > train_max)).any(axis=1)
top20 = top20.assign(outside_training_range=outside.values)
display(top20[['COF_ID','predicted_CO2_uptake_demo','outside_training_range']])


## 7. 解释：什么特征推动了筛选结果？

公开 COF screening 工作使用 SHAP。初学阶段先从 Random Forest feature importance 看起，再在后续项目升级到 SHAP。

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
plt.figure(figsize=(6,4))
importance.plot(kind='barh')
plt.xlabel('Random-forest feature importance')
plt.show()


## 8. 真正科研项目还缺哪些步骤？

1. 从 CIF 或专门软件生成可复现 descriptors；
2. 明确 target 条件，例如 CO₂ uptake 的温度、压力、force field/GCMC protocol；
3. 保存 `COF_ID ↔ CIF ↔ feature row ↔ target row`；
4. 去除重复/近重复结构；
5. 设计 random split 以外的 family/topology-aware split；
6. 做超参数选择时避免 test leakage；
7. 用 SHAP/partial dependence 等分析，而不是只报告 R²；
8. 对 top candidates 回到 CIF 检查结构合理性；
9. 对最终候选重新进行高精度模拟或实验验证；
10. 保存模型版本、descriptor 版本与筛选条件。

机器学习筛选的输出应当是**待验证的候选材料**，而不是“模型已经发现了最佳 COF”。

## 9. 推荐继续阅读的仓库路线

- **CURATED-COFs**：真实实验 COF CIF 与结构清理记录。
- **CoRE-COF Database**：更大规模 COF 数据库与版本化结构筛选。
- **SupportingInformation_CO2captureHTS_2024**：CO₂ capture 的高通量 + ML + SHAP 路线。
- **mofdscribe**：虽然主要面向 MOF/多孔材料，但 featurization、benchmark 和 splitting 思想值得迁移到 COF。

不要直接复制仓库脚本。学习重点是识别可复用的科研结构：**data generation → representation → validation → screening → interpretation → verification**。

## Exercises

1. 把 reference 数量改成 40、80、200，比较 test MAE 和 top-20 稳定性。
2. 删除 pore descriptors，只保留 composition；再反过来只保留 pore descriptors。
3. 为 `top20` 设计一个第二阶段昂贵计算队列。
4. 解释为什么不能反复用同一个 test set 调模型又报告它作为最终性能。
5. 写出自己的真实项目目录：`cifs/`、`features/`、`targets/`、`splits/`、`models/`、`predictions/`。

### Level A 真正完成标准

你不仅会 `model.fit(X, y)`，还能够解释：**结构从哪里来 → descriptor 怎么生成 → target 怎么定义 → 数据怎么对齐 → split 怎么设计 → 模型如何验证 → 如何筛选未知 COF → 为什么还必须回到结构验证。**
